# BankToBook — Phased Regression Harness

**Issue #40 — P0 Baseline (B1–B5)**

Baseline using existing `llcBankView` / `llcExpRev` — no agents.  
Invariant: trial balance total = 0 at every phase.

| Cell | Purpose |
|------|------|
| B1 | Setup + config |
| B2 | Parse 2025 WF CSVs → bank DataFrame |
| B3 | Load llcExpRev (53 already-booked records) → ExpRev DataFrame |
| B4 | Double-entry GL expansion → GL DataFrame |
| B5 | Trial balance (Debit total − Credit total must = 0) |

P1+ cells will be added below as IngestAgent and BankAgent phases are built.

In [1]:
# B1 — Setup & Config
import sys
import json
import datetime
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

REPO = Path.cwd().parent          # llcRentalTracker/
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from ledger import setup_paths
setup_paths.load_config('WBGroupLLC', 2025)

from ledger.LLC import LLC
llc = LLC('WBGroupLLC')

print("BUS root    :", setup_paths.TOP)
print("BankStmts   :", setup_paths.BANK_STMTS)
print("Accts dir   :", setup_paths.ACCTS_DIR)
print("Year        :", setup_paths.YEAR)

[setup_paths] Loaded 'WBGroupLLC/2025' from /Users/frankrojas/.llcRentalTracker/config.json → bus_repo=/Users/frankrojas/Library/CloudStorage/GoogleDrive-frankr6591@gmail.com/My Drive/Family/Assets/LLC-WBGroup
BUS root    : /Users/frankrojas/Library/CloudStorage/GoogleDrive-frankr6591@gmail.com/My Drive/Family/Assets/LLC-WBGroup
BankStmts   : /Users/frankrojas/Library/CloudStorage/GoogleDrive-frankr6591@gmail.com/My Drive/Family/Assets/LLC-WBGroup/books/2025/BankStmts
Accts dir   : /Users/frankrojas/Library/CloudStorage/GoogleDrive-frankr6591@gmail.com/My Drive/Family/Assets/LLC-WBGroup/books/Accts
Year        : 2025


In [ ]:
# B2 — Parse 2025 YE WF bank statement → bank DataFrame
#
# 2025 YE statement is WBGroupLLC_WF_20251231.csv — this supersedes the two
# earlier partial statements (20251211, 20251216) which are subsets of it.
# Using only the YE file avoids duplicate transactions from partial exports.
from ui.llcBankView import _parse_wf_csv

bank_dir = setup_paths.BANK_STMTS
ye_csv   = bank_dir / 'WBGroupLLC_WF_20251231.csv'

print(f"YE statement : {ye_csv.name}")
if not ye_csv.exists():
    raise FileNotFoundError(f"YE bank statement not found: {ye_csv}")

with open(ye_csv, 'r', encoding='utf-8', errors='replace') as fh:
    bank_rows = _parse_wf_csv(fh.read())

bank_df = pd.DataFrame(bank_rows)
print(f"Rows parsed  : {len(bank_df)}")
display(bank_df.head(5))

In [3]:
# B3 — Load llcExpRev (already-booked 2025 records) → ExpRev DataFrame
from ledger.llcExpRev import llcExpRev

er_obj = llcExpRev(llc)
er_records = er_obj.load()          # always returns list (unwraps new dict format)
er_log     = er_obj.log_history()   # LogHistory for audit trail

print(f"ExpRev records : {len(er_records)}")
print(f"LogHistory entries : {len(er_log)}")

er_df = pd.DataFrame(er_records)
print(f"\nAccounts (acct):\n  {sorted(er_df['acct'].unique())}")
print(f"\nrefDB values: {sorted(er_df['refDB'].unique())}")
display(er_df[['dt', 'acct', 'Ledger', 'aType', 'amt', 'desc', 'propNm', 'refDB']].head(5))

ExpRev records : 53
LogHistory entries : 0

Accounts (acct):
  ['Acct.Cash.Bank']

refDB values: ['llcBank-Manual']


,dt,acct,Ledger,aType,amt,desc,propNm,refDB
0,2025.08.20,Acct.Cash.Bank,Acct.Equity.Owner.Capital.Funds,Debit,219000.00,Owner investment,H_805HighMesa,llcBank-Manual
1,2025.08.20,Acct.Cash.Bank,Acct.Equity.Owner.Capital.Funds,Debit,50.00,Owner Investment,H_805HighMesa,llcBank-Manual
2,2025.08.26,Acct.Cash.Bank,Acct.Cash.Escrow,Credit,213936.95,Property Purchase,H_805HighMesa,llcBank-Manual
3,2025.08.28,Acct.Cash.Bank,Acct.Exp.Util,Credit,250.00,Auto: Pay Monthly Util,H_805HighMesa,llcBank-Manual
4,2025.08.28,Acct.Cash.Bank,Acct.Exp.Other,Credit,75.76,NoRcp: Approved Purchase: AMAZON MKTPL*6Y5UJ A...,H_805HighMesa,llcBank-Manual


In [4]:
# B4 — Double-entry GL expansion
#
# Each llcExpRev record has two account legs:
#   acct   — primary account (e.g. Acct.Cash.Bank)
#   Ledger — contra account (e.g. Acct.Equity.Owner.Capital.Funds)
#
# Expansion: 2 GL rows per source record.
#   Side A: acct=acct,   aType=original aType
#   Side B: acct=Ledger, aType=flipped

def to_double_entry(records):
    df = pd.DataFrame(records) if not isinstance(records, pd.DataFrame) else records.copy()
    # Drop rows with missing Ledger (not a full dual-entry record)
    df = df[df['Ledger'].notna() & ~df['Ledger'].isin(['', 'nan'])].copy()

    side_a = df.copy()
    side_a['_side'] = 'A'

    side_b = df.copy()
    side_b['acct'] = side_b['Ledger']
    side_b['aType'] = side_b['aType'].apply(lambda v: 'Credit' if str(v).strip().lower() in ('debit', 'dr') else 'Debit')
    side_b['_side'] = 'B'

    gl = pd.concat([side_a, side_b], ignore_index=True)
    gl = gl.drop(columns=['Ledger', '_side'], errors='ignore')
    gl['signed_amt'] = gl.apply(
        lambda r: float(r['amt']) if str(r['aType']).strip().lower() in ('debit', 'dr') else -float(r['amt']),
        axis=1
    )
    return gl.sort_values('dt').reset_index(drop=True)


er_gl_df = to_double_entry(er_records)
print(f"Source records : {len(er_records)}")
print(f"GL rows (×2)   : {len(er_gl_df)}")
display(er_gl_df[['dt', 'acct', 'aType', 'amt', 'signed_amt', 'desc']].head(8))

Source records : 53
GL rows (×2)   : 106


,dt,acct,aType,amt,signed_amt,desc
0,2025.08.20,Acct.Cash.Bank,Debit,219000.00,219000.00,Owner investment
1,2025.08.20,Acct.Cash.Bank,Debit,50.00,50.00,Owner Investment
2,2025.08.20,Acct.Equity.Owner.Capital.Funds,Credit,50.00,-50.00,Owner Investment
3,2025.08.20,Acct.Equity.Owner.Capital.Funds,Credit,219000.00,-219000.00,Owner investment
4,2025.08.26,Acct.Cash.Bank,Credit,213936.95,-213936.95,Property Purchase
5,2025.08.26,Acct.Cash.Escrow,Debit,213936.95,213936.95,Property Purchase
6,2025.08.28,Acct.Cash.Bank,Credit,250.00,-250.00,Auto: Pay Monthly Util
7,2025.08.28,Acct.Cash.Bank,Credit,75.76,-75.76,NoRcp: Approved Purchase: AMAZON MKTPL*6Y5UJ A...


In [5]:
# B5 — Trial Balance  (INVARIANT: net signed amount = 0)
#
# Debit  = positive (increases asset / expense accounts)
# Credit = negative (increases liability / equity / income accounts)
# Net must be 0 — any non-zero value is a double-entry bug.

debit_total  = er_gl_df.loc[er_gl_df['aType'].str.lower() == 'debit',  'amt'].sum()
credit_total = er_gl_df.loc[er_gl_df['aType'].str.lower() == 'credit', 'amt'].sum()
net          = round(debit_total - credit_total, 2)

print(f"Total Debits  : ${debit_total:>12,.2f}")
print(f"Total Credits : ${credit_total:>12,.2f}")
print(f"Net (D − C)   : ${net:>12,.2f}")

if abs(net) < 0.01:
    print("\n✓ TRIAL BALANCE = 0  —  books are balanced (P0 baseline PASS)")
else:
    print(f"\n✗ TRIAL BALANCE ERROR: ${net:,.2f}  —  double-entry bug, fix before P1")

# Summary by account type
try:
    from ledger.llcCOA import ChartOfAccounts
    coa = ChartOfAccounts(llc)
    er_gl_df['acctType'] = er_gl_df['acct'].apply(lambda a: coa._Type(a) if a else '')
except Exception as e:
    print(f"(COA lookup skipped: {e})")
    er_gl_df['acctType'] = er_gl_df['acct'].apply(
        lambda a: 'Expense' if 'Exp' in str(a)
        else ('Income' if 'Rev' in str(a)
        else ('Asset' if ('Cash' in str(a) or 'Fixed' in str(a))
        else ('Equity' if 'Equity' in str(a)
        else 'Other')))
    )

tb = (
    er_gl_df.groupby(['acctType', 'aType'])['amt']
    .sum()
    .unstack(fill_value=0)
)
for col in ['Debit', 'Credit']:
    if col not in tb:
        tb[col] = 0
tb['Balance'] = tb['Debit'] - tb['Credit']
tb.loc['TOTAL'] = tb.sum()

print("\nTrial Balance by Account Type:")
display(tb.style.format('${:,.2f}'))

Total Debits  : $  440,577.19
Total Credits : $  440,577.19
Net (D − C)   : $       -0.00

✓ TRIAL BALANCE = 0  —  books are balanced (P0 baseline PASS)

Trial Balance by Account Type:


aType,Credit,Debit,Balance
acctType,,,
Asset,"$216,905.60","$438,868.62","$221,963.02"
Equity,"$219,257.00",$0.00,"$-219,257.00"
Expense,$14.06,"$1,708.04","$1,693.98"
Income,"$4,400.53",$0.53,"$-4,400.00"
TOTAL,"$440,577.19","$440,577.19",$-0.00


In [ ]:
# B6 — 2025 Full-GL Baseline Assertions  (PA-verified reference values)
#
# These assert the FULL books baseline (all 4 source DBs) against values
# verified on PythonAnywhere before the P0 schema migration.
# Any regression in the core accounting pipeline will trip an assert here.
#
# GL TOTAL   : 672,945.93 / 672,945.93 / 0.00
# IS         : total_income=4400, net_rental=667.55, subtotal_rental_expense=3732.45, depreciation=1903.13
# BS         : D=669,198.89  C=668,531.34  B=667.55  (balance-sheet accts only)

from ledger.stmtGL import stmtGL, stmtGL_View
from ledger.stmtIS import stmtIS
from ledger.stmtBS import stmtBS

# ── Full 4-source GL ─────────────────────────────────────────────────────────
full_gl      = stmtGL(llc)
full_gl_rows = full_gl._rows

gl_debit  = sum(r['amt'] for r in full_gl_rows if str(r.get('aType','')).lower() in ('debit','dr') and r.get('refDB') != 'COA')
gl_credit = sum(r['amt'] for r in full_gl_rows if str(r.get('aType','')).lower() in ('credit','cr') and r.get('refDB') != 'COA')

print(f"GL Debits  : {gl_debit:>12,.2f}   (expected 672,945.93)")
print(f"GL Credits : {gl_credit:>12,.2f}   (expected 672,945.93)")
assert abs(gl_debit  - 672945.93) < 0.02, f"GL Debit mismatch: {gl_debit}"
assert abs(gl_credit - 672945.93) < 0.02, f"GL Credit mismatch: {gl_credit}"
assert abs(gl_debit  - gl_credit) < 0.02, f"GL not balanced: {gl_debit - gl_credit}"
print("✓ GL total")

# ── Income Statement ─────────────────────────────────────────────────────────
is_agg = stmtIS(llc, gl_records=full_gl_rows).taxAggregates()

print(f"\nIS total_income              : {is_agg['total_income']:>10,.2f}   (expected  4,400.00)")
print(f"IS net_rental                : {is_agg['net_rental']:>10,.2f}   (expected    667.55)")
print(f"IS subtotal_rental_expense   : {is_agg['subtotal_rental_expense']:>10,.2f}   (expected  3,732.45)")
print(f"IS depreciation              : {is_agg['depreciation']:>10,.2f}   (expected  1,903.13)")

_net_rental_before_depr = is_agg['subtotal_rental_income'] - (is_agg['subtotal_rental_expense'] - is_agg['depreciation'])
print(f"IS net_rental_before_depr    : {_net_rental_before_depr:>10,.2f}   (expected  2,570.68)")

assert abs(is_agg['total_income']            -  4400.00) < 0.02, f"IS total_income: {is_agg['total_income']}"
assert abs(is_agg['net_rental']              -   667.55) < 0.02, f"IS net_rental: {is_agg['net_rental']}"
assert abs(is_agg['subtotal_rental_expense'] -  3732.45) < 0.02, f"IS subtotal_rental_expense: {is_agg['subtotal_rental_expense']}"
assert abs(is_agg['depreciation']            -  1903.13) < 0.02, f"IS depreciation: {is_agg['depreciation']}"
assert abs(_net_rental_before_depr           -  2570.68) < 0.02, f"IS net_rental_before_depr: {_net_rental_before_depr}"
print("✓ IS aggregates")

# ── Balance Sheet (GL-level BS-account trial balance) ────────────────────────
_BS_TYPES = {'Asset', 'Liability', 'Equity'}
bs_debit  = sum(r['amt'] for r in full_gl_rows if r.get('acctType') in _BS_TYPES and str(r.get('aType','')).lower() in ('debit','dr')  and r.get('refDB') != 'COA')
bs_credit = sum(r['amt'] for r in full_gl_rows if r.get('acctType') in _BS_TYPES and str(r.get('aType','')).lower() in ('credit','cr') and r.get('refDB') != 'COA')

print(f"\nBS Debit   : {bs_debit:>12,.2f}   (expected 669,198.89)")
print(f"BS Credit  : {bs_credit:>12,.2f}   (expected 668,531.34)")
print(f"BS Balance : {round(bs_debit-bs_credit,2):>12,.2f}   (expected     667.55)")

assert abs(bs_debit  - 669198.89) < 0.02, f"BS Debit mismatch: {bs_debit}"
assert abs(bs_credit - 668531.34) < 0.02, f"BS Credit mismatch: {bs_credit}"
assert abs(bs_debit - bs_credit   -    667.55) < 0.02, f"BS Balance mismatch: {bs_debit - bs_credit}"
print("✓ BS trial balance")

print("\n=== B6 PASS — 2025 baseline verified ===")


---
## P1 cells — IngestAgent compat check
_To be added after IngestAgent (BkVendorKB + BkTxnTypeDetector) is implemented._

- **P1a**: IngestAgent compat check — 2025 classify diff vs B3 baseline (expected: zero diff for Tier 1 rules)
- **P1b**: IngestAgent 2026 classify — new transactions

## P2 cells — BankAgent compat check
_To be added after BankAgent (BankCSVParser + BkDuplicateDetector + BkCIPGuard) is implemented._

- **P2a**: BankAgent 2025 compat — trial-balance delta vs B5 must be 0 for non-CIP rows
- **P2b**: BankAgent 2026 preview
- **P3a**: 2026 GL + trial balance
- **P3b**: optional commit cell (commented out by default)